
<a id='brainscales-2-single-neuron-experiments'></a>

# BrainScaleS-2 single neuron experiments

In order to use the microscheduler we have to set some environment variables first:

In [ ]:
from _static.common.helpers import setup_hardware_client

setup_hardware_client()

In [ ]:
%matplotlib inline
from functools import partial

import numpy as np
from ipywidgets import IntSlider
from ipywidgets import interact

IntSlider = partial(IntSlider, continuous_update=False)
import matplotlib.pyplot as plt

plt.style.use("_static/matplotlibrc")

import pynn_brainscales.brainscales2 as pynn
from pynn_brainscales.brainscales2 import Population
from pynn_brainscales.brainscales2.standardmodels.cells import SpikeSourceArray
from pynn_brainscales.brainscales2.standardmodels.synapses import StaticSynapse


def plot_membrane_dynamics(population: Population, segment_id=-1, ylim=None):
    """
    Plot the membrane potential of the neuron in a given population view. Only
    population views of size 1 are supported.
    :param population: Population, membrane traces and spikes are plotted for.
    :param segment_id: Index of the neo segment to be plotted. Defaults to
                       -1, encoding the last recorded segment.
    :param ylim: y-axis limits for the plot.
    """
    if len(population) != 1:
        raise ValueError("Plotting is supported for populations of size 1.")
    # Experimental results are given in the 'neo' data format
    mem_v = population.get_data("v").segments[segment_id].irregularlysampledsignals[0]

    plt.plot(mem_v.times, mem_v, alpha=0.5)
    print(f"Mean membrane potential: {mem_v.mean()}")
    plt.xlabel("Wall clock time [ms]")
    plt.ylabel("ADC readout [a.u.]")
    if ylim:
        plt.ylim(ylim)

In this part we will explore some of the spiking capabilities of the
BrainScaleS-2 neuromorphic accelerator using our implementation of the
pyNN interface.

There are 512 neuron circuits emulating the adaptive exponential
integrate-and-fire model model and 131,072 STP/STDP synapses in one
HICANN-X chip, but we will stick to a single neuron for now.

## A silent neuron

As a first experiment, we will record the membrane of a single, silent
neuron on the analog substrate. We use pyNN as an interface, which can
be used similarly to existing simulators and other neuromorphic
platforms.

While we provide calibrated neurons that have their hardware parameters
set to match high-level targets, e.g., time-constants, we will start
with modifying a few circuit parameters directly. These control the
dynamic behavior of the neuron as well as static configuration. Most of
them are either boolean or given in units of ‘LSB’ for chip-internal
Digital-to-Analog converters, providing voltages and currents - they
have no direct biological translation.

For this first example, you may alter the leak potential via the slider
and observe the response of the analog neuron’s resting potential.

We first define a population of one neuron and let its membrane
potential be recorded. The experiment is executed by calling
`pynn.run(time_in_ms)`. The network is evolved for a given amount of
time and neurons are stimulated by any given stimuli.

The time is given in units of milliseconds (wall clock time),
representing the hardware’s intrinsic 1000-fold speed-up compared to
biological systems.

In [ ]:
@interact(v_leak=IntSlider(min=400, max=1022, step=1, value=700))
def experiment(v_leak):
    plt.figure()
    plt.title("First experiment: A silent neuron")

    pynn.setup()

    pop = pynn.Population(
        1,
        pynn.cells.HXNeuron(
            # Leak potential, range: 400-1000
            leak_v_leak=v_leak,
            # Leak conductance, range: 0-1022
            leak_i_bias=1022,
        ),
    )

    # The chip contains a fast Analog-to-Digital converter. It can be used to
    # record different observables of a single analog neuron - most importantly
    # the membrane potential.
    pop.record(["v"])

    # Execute experiment
    pynn.run(0.2)

    plot_membrane_dynamics(pop, ylim=(100, 800))
    plt.show()

    # Reset the pyNN internal state and prepare for the following experiment.
    pynn.end()

## Leak over threshold

As a second experiment, we will let the neurons on BrainScaleS-2 spike
by setting a ‘leak-over-threshold’ configuration. The leak potential is
set high, above the spike threshold, so that the membrane charges
exponentially until a spike is triggered. The potential is then reset to
a lower voltage.

The parametrization of the different potentials is not equal, a lower
threshold setting of 300 may correspond to a higher leak potential of
700.

In [ ]:
@interact(
    v_leak=IntSlider(min=400, max=1022, step=1, value=1000),
    v_threshold=IntSlider(min=0, max=500, step=1, value=300),
    v_reset=IntSlider(min=300, max=1022, step=1, value=400),
    i_bias_leak=IntSlider(min=0, max=1022, step=1, value=150),
)
def experiment(v_leak, v_threshold, v_reset, i_bias_leak):
    """
    Set up a leak over threshold neuron.

    :param v_leak: Leak potential.
    :param v_threshold: Spike threshold potential.
    :param v_reset: Reset potential.
    :param i_bias_leak: Controls the leak conductance (membrane time constant).
    """

    plt.figure()
    plt.title("Second experiment: Leak over threshold")

    pynn.setup()

    pop = pynn.Population(
        1,
        pynn.cells.HXNeuron(
            # Leak potential, range: 400-1000
            leak_v_leak=v_leak,
            # Leak conductance, range: 0-1022
            leak_i_bias=i_bias_leak,
            # Threshold potential, range: 0-500
            threshold_v_threshold=v_threshold,
            # Reset potential, range: 300-1000
            reset_v_reset=v_reset,
            # Membrane capacitance, range: 0-63
            membrane_capacitance_capacitance=63,
            # Refractory time (counter), range: 0-255
            refractory_period_refractory_time=255,
            # Enable reset on threshold crossing
            threshold_enable=True,
            # Reset conductance, range: 0-1022
            reset_i_bias=1022,
            # Increase reset conductance
            reset_enable_multiplication=True,
        ),
    )

    pop.record(["v", "spikes"])
    pynn.run(0.2)
    plot_membrane_dynamics(pop, ylim=(100, 800))
    plt.show()
    pynn.end()

The neuron has many more parameters you may play around with.
Some documentation for these parameters is available in our [Lower-level API documentation](https://electronicvisions.github.io/documentation-brainscales2/api_lola.html).
The kewords you set in PyNN are generated from a hierarchical structure – search for the last part of a parameter (e.g. `capacitance`).

In [ ]:
pynn.cells.HXNeuron().get_parameter_names()

## Fixed-pattern variations

Due to the analog nature of the BrainScaleS-2 platform, the inevitable
mismatch of semiconductor fabrication results in inhomogeneous
properties of the computational elements.

We will visualize these effects by recording the membrane potential of
multiple neurons in leak-over-threshold configuration. You will notice
different resting, reset and threshold potentials as well as varying
membrane time constants.

In [ ]:
plt.figure()
plt.title("Third experiment: Fixed-pattern variations")

pynn.setup()

pop = pynn.Population(
    10,
    pynn.cells.HXNeuron(
        # Leak potential, range: 400-1000
        leak_v_leak=1000,
        # Leak conductance, range: 0-1022
        leak_i_bias=200,
        # Threshold potential, range: 0-600
        threshold_v_threshold=500,
        # Reset potential, range: 300-1000
        reset_v_reset=400,
        # Membrane capacitance, range: 0-63
        membrane_capacitance_capacitance=63,
        # Refractory time, range: 0-255
        refractory_period_refractory_time=255,
        # Enable reset on threshold crossing
        threshold_enable=True,
        # Reset conductance, range: 0-1022
        reset_i_bias=1022,
        # Increase reset conductance
        reset_enable_multiplication=True,
    ),
)

for neuron_id in range(len(pop)):
    print(f"Recording fixed-pattern variations: Run {neuron_id}")
    p_view = pynn.PopulationView(pop, [neuron_id])
    p_view.record(["v"])
    pynn.run(0.1)
    plot_membrane_dynamics(p_view, ylim=(100, 800))
    pynn.reset()
    pop.record(None)

plt.show()
pynn.end()

The plot shows the recorded membrane traces of multiple different
neurons. Due to the time-continuous nature of the system, there is no
temporal alignment between the individual traces, so the figure shows
multiple independent effects:

- Temporal misalignment: From the system’s view, the recording happens
  in an arbitrary time frame during the continuously evolving
  integration. Neurons are not synchronized to each other.  
- Circuit-level mismatch: Each individual neurons shows slightly
  different analog properties. The threshold is different for all
  traces; as is the membrane time constant (visible as slope) and the
  reset potentials (visible as plateaus during the refractory time).  

## Summary

The analog neurons on BrainScaleS-2 feature many hardware parameters
that can be set to achieve different operating points and to equalize
the behaviour of different neurons.

We will employ an automated calibration to get the neurons in the
desired operating mode. * In the next notebook, we will show spiking
operation and learning. * Later, we will change the target parameters
and use integrator neurons for executing multiply-accumulate operations.

## Outlook: External stimulation

We will continue the tutorial in the next notebook using external
stimulation on the neurons. You may play around a bit already here, but
we will revisit the following later.

Up to now, we have observed analog neurons without external stimulus. In
this experiment, we will introduce the latter and examine post-synaptic
pulses on the analog neuron’s membrane.

Preparing the neuron to receive synaptic inputs requires the
configuration of additional circuits. The additional settings include
technical parameters for bringing the circuit to its designed operating
point as well as configuration with a direct biological equivalent. For
simplicity, we will turn to a calibration which yields all required
parameters and counters the fixed pattern noise between different
neurons observed previously.

We represent projections as entries in the synapse matrix on the
neuromorphic chip. Weights are stored in digital 6bit values (plus
sign), the value range for on-chip weights is therefore -63 to 63. With
this first projection, we connect the external spike source to the
observed on-chip neuron population.

A default calibration is generated for every setup every night.
We save the nightly calibration in a variable such that we can use it later when we define our experiment.

In [ ]:
from _static.common.helpers import get_nightly_calibration

calib = get_nightly_calibration()

Now we can continue by defining our experiment:

In [ ]:
@interact(
    exc_weight=IntSlider(min=0, max=63, step=1, value=31),
    inh_weight=IntSlider(min=0, max=63, step=1, value=31),
    isi=IntSlider(min=10, max=100, step=5, value=50),
)
def run_experiment(exc_weight: int, inh_weight: int, isi: float):
    """
    Run external input demonstration on BSS-2.

    Adjust weight of projections, set input spikes and execute experiment
    on BSS-2.

    :param exc_weight: Weight of excitatory synaptic inputs, value range
        [0,63].
    :param inh_weight: Weight of inhibitory synaptic inputs, value range
        [0,63].
    :param isi: Time between synaptic inputs in microsecond (hardware
        domain)
    """

    plt.figure()
    plt.title("Fourth experiment: External stimulation")

    pynn.setup(initial_config=calib)

    # use calibrated parameters for neuron
    stimulated_p = pynn.Population(1, pynn.cells.HXNeuron())
    stimulated_p.record(["v", "spikes"])

    # calculate spike times
    wait_before_experiment = 0.01  # ms (hw)
    isi_ms = isi / 1000  # convert to ms
    spiketimes = np.arange(6) * isi_ms + wait_before_experiment

    # all but one input are chosen to be exciatory
    excitatory_spike = np.ones_like(spiketimes, dtype=bool)
    excitatory_spike[1] = False

    # external input
    exc_spikes = spiketimes[excitatory_spike]
    exc_stim_pop = pynn.Population(2, SpikeSourceArray(spike_times=exc_spikes))
    exc_proj = pynn.Projection(
        exc_stim_pop,
        stimulated_p,
        pynn.AllToAllConnector(),
        synapse_type=StaticSynapse(weight=exc_weight),
        receptor_type="excitatory",
    )

    inh_spikes = spiketimes[~excitatory_spike]
    inh_stim_pop = pynn.Population(2, SpikeSourceArray(spike_times=inh_spikes))
    inh_proj = pynn.Projection(
        inh_stim_pop,
        stimulated_p,
        pynn.AllToAllConnector(),
        synapse_type=StaticSynapse(weight=-inh_weight),
        receptor_type="inhibitory",
    )

    # run experiment
    pynn.run(0.6)
    plot_membrane_dynamics(stimulated_p, ylim=(100, 600))
    plt.show()
    pynn.end()

In [ ]:
%matplotlib inline
from _static.common.helpers import setup_hardware_client

setup_hardware_client()

import matplotlib.pyplot as plt
import pynn_brainscales.brainscales2 as pynn
from _static.tutorial.snn_adex_helpers import build_gui
from _static.tutorial.snn_adex_helpers import plot_membrane_dynamics
from ipywidgets import fixed

In [ ]:
import pyhalco_hicann_dls_vx_v3 as halco
import pylola_vx_v3 as lola


def get_initial_config():
    config = lola.Chip()
    for i in halco.iter_all(halco.CommonNeuronBackendConfigOnDLS):
        config.neuron_block.backends[i].enable_clocks = True
        config.neuron_block.backends[i].clock_scale_fast = 3
        config.neuron_block.backends[i].clock_scale_slow = 3
        config.neuron_block.backends[i].clock_scale_adaptation_pulse = 7
        config.neuron_block.backends[i].clock_scale_post_pulse = 5
    return config

You may play around with the parameters in this experiment to achieve
different traces. Try to stack multiple PSPs, try to make the neuron
spike more often, be creative!

In [ ]:
def experiment(**kwargs):
    global last_configuration
    last_configuration = kwargs

    target_neuron = kwargs.pop("target_neuron", 0)

    pynn.setup(
        neuronPermutation=[target_neuron, target_neuron + 1],
        initial_config=get_initial_config(),
    )

    pop = pynn.Population(2, pynn.cells.HXNeuron())

    neuron = pop[0]
    readout_dummy = pop[1]

    neuron.reset_v_reset = 300
    neuron.reset_i_bias = 1022
    neuron.reset_enable_multiplication = True
    neuron.membrane_capacitance_capacitance = 63
    neuron.refractory_period_refractory_time = 30
    neuron.refractory_period_enable_pause = True

    neuron.threshold_v_threshold = 620
    neuron.threshold_enable = True

    # adaptation
    neuron.adaptation_enable_pulse = True
    neuron.adaptation_i_bias_b = 0

    neuron.multicompartment_connect_right = True

    neuron.exponential_enable = any([k.startswith("exponential") for k in kwargs])

    if any([k.startswith("adaptation") for k in kwargs]):
        enable_adaptation = False
        if "adaptation_i_bias_a" in kwargs and kwargs["adaptation_i_bias_a"] != 0:
            enable_adaptation = True
        if "adaptation_i_bias_b" in kwargs and kwargs["adaptation_i_bias_b"] != 0:
            enable_adaptation = True
        neuron.adaptation_enable = enable_adaptation

        if enable_adaptation:
            # the adaptation leak reference has to track the membrane leak to avoid offset currents
            neuron.adaptation_v_leak = 500 + kwargs["leak_v_leak"] - 700

    for k, v in kwargs.items():
        setattr(neuron, k, v)

    readout_dummy.membrane_capacitance_capacitance = 0
    readout_dummy.leak_i_bias = 0
    readout_dummy.leak_enable_division = True
    readout_dummy.threshold_enable = False

    pop[0:1].record(["adaptation", "spikes"])
    pop[1:2].record(["v"])

    # schedule and execute hardware run
    pop[0:1].set(constant_current_enable=False)
    pynn.run(0.1, pynn.RunCommand.APPEND)
    pop[0:1].set(constant_current_enable=True)
    pynn.run(0.6, pynn.RunCommand.APPEND)
    pop[0:1].set(constant_current_enable=False)
    pynn.run(0.3, pynn.RunCommand.APPEND)
    pop[0:1].set(constant_current_enable=True)
    pynn.run(0.1, pynn.RunCommand.APPEND)
    pop[0:1].set(constant_current_enable=False)
    pynn.run(0.1, pynn.RunCommand.EXECUTE)

    plot_membrane_dynamics(pop)
    plt.show()

    pynn.end()

In [ ]:
build_gui(
    experiment,
    [
        "leak_i_bias",
        "leak_v_leak",
        "adaptation_v_ref",
        "adaptation_i_bias_tau",
        "adaptation_i_bias_a",
        "constant_current_i_offset",
    ],
    {"target_neuron": fixed(1)},
    defaults={"adaptation_i_bias_a": 10, "leak_enable_division": True},
)

In [ ]:
def experiment(**kwargs):
    global last_configuration
    last_configuration = kwargs

    target_neuron = kwargs.pop("target_neuron", 0)

    pynn.setup(
        neuronPermutation=[target_neuron, target_neuron + 1],
        initial_config=get_initial_config(),
    )

    pop = pynn.Population(2, pynn.cells.HXNeuron())

    neuron = pop[0]
    readout_dummy = pop[1]

    neuron.reset_v_reset = 300
    neuron.reset_i_bias = 1022
    neuron.reset_enable_multiplication = True
    neuron.membrane_capacitance_capacitance = 63
    neuron.refractory_period_refractory_time = 30
    neuron.refractory_period_enable_pause = True

    neuron.threshold_v_threshold = 620
    neuron.threshold_enable = True

    # adaptation
    neuron.adaptation_enable_pulse = True
    neuron.adaptation_i_bias_b = 0

    neuron.multicompartment_connect_right = True

    neuron.exponential_enable = any([k.startswith("exponential") for k in kwargs])

    if any([k.startswith("adaptation") for k in kwargs]):
        enable_adaptation = False
        if "adaptation_i_bias_a" in kwargs and kwargs["adaptation_i_bias_a"] != 0:
            enable_adaptation = True
        if "adaptation_i_bias_b" in kwargs and kwargs["adaptation_i_bias_b"] != 0:
            enable_adaptation = True
        neuron.adaptation_enable = enable_adaptation

        if enable_adaptation:
            # the adaptation leak reference has to track the membrane leak to avoid offset currents
            neuron.adaptation_v_leak = 500 + kwargs["leak_v_leak"] - 700

    for k, v in kwargs.items():
        setattr(neuron, k, v)

    readout_dummy.membrane_capacitance_capacitance = 0
    readout_dummy.leak_i_bias = 0
    readout_dummy.leak_enable_division = True
    readout_dummy.threshold_enable = False

    pop[0:1].record(["adaptation", "spikes"])
    pop[1:2].record(["v"])

    # schedule and execute hardware run
    pop[0:1].set(constant_current_enable=False)
    pynn.run(0.1, pynn.RunCommand.APPEND)
    pop[0:1].set(constant_current_enable=True)
    pynn.run(0.6, pynn.RunCommand.APPEND)
    pop[0:1].set(constant_current_enable=False)
    pynn.run(0.3, pynn.RunCommand.APPEND)
    pop[0:1].set(constant_current_enable=True)
    pynn.run(0.1, pynn.RunCommand.APPEND)
    pop[0:1].set(constant_current_enable=False)
    pynn.run(0.1, pynn.RunCommand.EXECUTE)

    plot_membrane_dynamics(pop)
    plt.show()

    pynn.end()

In [ ]:
# Select physical neuron
target_neuron = 0
n_neurons = 10

pynn.setup(
    neuronPermutation=list(range(target_neuron, target_neuron + n_neurons)),
    initial_config=get_initial_config(),
)

pop = pynn.Population(n_neurons, pynn.cells.HXNeuron())

neuron = pop[0]

# Reset
neuron.reset_v_reset = 300
neuron.reset_i_bias = 1022
neuron.reset_enable_multiplication = True

# Membrane
neuron.membrane_capacitance_capacitance = 63
neuron.leak_v_leak = 700

# Refractory period
neuron.refractory_period_refractory_time = 30
neuron.refractory_period_enable_pause = True

# Threshold
neuron.threshold_v_threshold = 620
neuron.threshold_enable = True

# Adaptation
neuron.adaptation_enable_pulse = True
neuron.adaptation_i_bias_a = 0
neuron.adaptation_i_bias_b = 0

neuron.adaptation_enable = (
    neuron.adaptation_i_bias_a != 0 or neuron.adaptation_i_bias_b != 0
)

if neuron.adaptation_enable:
    neuron.adaptation_v_leak = 500 + neuron.leak_v_leak - 700

# Exponential spike initiation
neuron.exponential_enable = False

# Disable multicompartment coupling
neuron.multicompartment_connect_right = False

# Constant current magnitude (hardware DAC value)
neuron.constant_current_i_offset = 100

# Record voltage and spikes
pop[0:1].record(["v", "spikes"])

# Stimulation protocol
pop.set(constant_current_enable=False)
pynn.run(0.1, pynn.RunCommand.APPEND)

pop.set(constant_current_enable=True)
pynn.run(0.6, pynn.RunCommand.APPEND)

pop.set(constant_current_enable=False)
pynn.run(0.3, pynn.RunCommand.APPEND)

pop.set(constant_current_enable=True)
pynn.run(0.1, pynn.RunCommand.APPEND)

pop.set(constant_current_enable=False)
pynn.run(0.1, pynn.RunCommand.EXECUTE)


import matplotlib.pyplot as plt

# Get recorded data
data = pop.get_data()
v = data.segments[0].filter(name="v")[0]

# Extract time and voltage
t = v.times.rescale("ms").magnitude
voltage = v.magnitude[:, 0]

# Plot
plt.figure(figsize=(10, 4))
plt.plot(t, voltage)
plt.xlabel("Time (ms)")
plt.ylabel("Membrane voltage (hardware units)")
plt.grid(True)
plt.show()

pynn.end()

In [ ]:
# IMPORTANT: assign the neuron
neuron = pop[0]

# Membrane
neuron.membrane_capacitance_capacitance = 63
neuron.leak_v_leak = 480
neuron.leak_i_bias = 150
neuron.leak_enable_division = True

# Threshold and reset
neuron.threshold_enable = True
neuron.threshold_v_threshold = 600
neuron.reset_v_reset = 500
neuron.reset_i_bias = 1022
neuron.reset_enable_multiplication = True

# Refractory period
neuron.refractory_period_refractory_time = 255
neuron.refractory_period_enable_pause = True

# Disable adaptation and exponential current for initial testing
neuron.adaptation_enable = False
neuron.exponential_enable = False

# Current injection
neuron.constant_current_i_offset = 100

In [ ]:
# Select physical neuron
target_neuron = 0
n_neurons = 10

pynn.setup(
    neuronPermutation=list(range(target_neuron, target_neuron + n_neurons)),
    initial_config=get_initial_config(),
)

pop = pynn.Population(n_neurons, pynn.cells.HXNeuron())

neuron.reset_v_reset = 300
neuron.reset_i_bias = 1022
neuron.reset_enable_multiplication = True
neuron.membrane_capacitance_capacitance = 63
neuron.refractory_period_refractory_time = 30
neuron.refractory_period_enable_pause = True

neuron.threshold_v_threshold = 620
neuron.threshold_enable = True

# adaptation
neuron.adaptation_enable_pulse = True
neuron.adaptation_i_bias_b = 0

neuron.multicompartment_connect_right = True


# Record voltage and spikes
pop[0:1].record(["v", "spikes"])

# Stimulation protocol
pop.set(constant_current_enable=False)
pynn.run(0.1, pynn.RunCommand.APPEND)

pop.set(constant_current_enable=True)
pynn.run(0.6, pynn.RunCommand.APPEND)

pop.set(constant_current_enable=False)
pynn.run(0.3, pynn.RunCommand.APPEND)

pop.set(constant_current_enable=True)
pynn.run(0.1, pynn.RunCommand.APPEND)

pop.set(constant_current_enable=False)
pynn.run(0.1, pynn.RunCommand.EXECUTE)


import matplotlib.pyplot as plt

# Get recorded data
data = pop.get_data()
v = data.segments[0].filter(name="v")[0]

# Extract time and voltage
t = v.times.rescale("ms").magnitude
voltage = v.magnitude[:, 0]

# Plot
plt.figure(figsize=(10, 4))
plt.plot(t, voltage)
plt.xlabel("Time (ms)")
plt.ylabel("Membrane voltage (hardware units)")
plt.grid(True)
plt.show()

pynn.end()

In [ ]:
import matplotlib.pyplot as plt

# Parameters (hardware DAC values)
v_leak = 1000
v_threshold = 300
v_reset = 400
i_bias_leak = 150

# Initialize BrainScaleS-2
pynn.setup()

# Create and configure one neuron
pop = pynn.Population(
    1,
    pynn.cells.HXNeuron(
        leak_v_leak=v_leak,
        leak_i_bias=i_bias_leak,
        threshold_v_threshold=v_threshold,
        reset_v_reset=v_reset,
        membrane_capacitance_capacitance=63,
        refractory_period_refractory_time=255,
        threshold_enable=True,
        reset_i_bias=1022,
        reset_enable_multiplication=True,
    ),
)

# Record membrane voltage and spikes
pop.record(["v", "spikes"])

# Run experiment
pynn.run(0.2)

# Retrieve recorded data before pynn.end()
data = pop.get_data()
segment = data.segments[0]

# Membrane voltage
v = segment.filter(name="v")[0]
t = v.times.rescale("ms").magnitude
voltage = v.magnitude[:, 0]

# Spike times
spikes = segment.spiketrains[0].rescale("ms").magnitude

# Plot
plt.figure(figsize=(10, 4))
plt.plot(t, voltage, label="Membrane voltage")

for spike_time in spikes:
    plt.axvline(spike_time, color="red", linestyle="--", alpha=0.4)

plt.xlabel("Time (ms)")
plt.ylabel("Membrane voltage (ADC units)")
plt.title("Leak-over-threshold neuron")
plt.ylim(100, 800)
plt.grid(True)
plt.legend()
plt.show()

print("Number of spikes:", len(spikes))

# Shut down simulator
pynn.end()

In [ ]:
import matplotlib.pyplot as plt

# Parameters (hardware DAC values)
v_leak = 1000
v_threshold = 300
v_reset = 400
i_bias_leak = 150

# Initialize BrainScaleS-2
pynn.setup()

# Create and configure one neuron
pop = pynn.Population(
    1,
    pynn.cells.HXNeuron(
        leak_v_leak=v_leak,
        leak_i_bias=i_bias_leak,
        threshold_v_threshold=v_threshold,
        reset_v_reset=v_reset,
        membrane_capacitance_capacitance=63,
        refractory_period_refractory_time=255,
        threshold_enable=True,
        reset_i_bias=1022,
        reset_enable_multiplication=True,
    ),
)

# Record membrane voltage and spikes
pop.record(["v", "spikes"])

# Run experiment
# pynn.run(0.3)

pop.set(constant_current_i_offset=1000)

pop.set(constant_current_enable=True)
pynn.run(0.35, pynn.RunCommand.APPEND)

pop.set(constant_current_enable=True)
pynn.run(0.25, pynn.RunCommand.APPEND)

# Execute both scheduled segments
pynn.run(None, pynn.RunCommand.EXECUTE)

# Stimulation protocol
# pop.set(constant_current_enable=False)
# pynn.run(0.05, pynn.RunCommand.APPEND)

# pop.set(constant_current_enable=True)
# pynn.run(0.05, pynn.RunCommand.APPEND)

# pop.set(constant_current_enable=False)
# pynn.run(0.05, pynn.RunCommand.APPEND)

# pop.set(constant_current_enable=True)
# pynn.run(0.05, pynn.RunCommand.APPEND)

# pop.set(constant_current_enable=False)
# pynn.run(0.1, pynn.RunCommand.EXECUTE)


# Retrieve recorded data before pynn.end()
data = pop.get_data()
segment = data.segments[0]

# Membrane voltage
v = segment.filter(name="v")[0]
t = v.times.rescale("ms").magnitude
voltage = v.magnitude[:, 0]

# Spike times
spikes = segment.spiketrains[0].rescale("ms").magnitude

# Plot
plt.figure(figsize=(10, 4))
plt.plot(t, voltage, label="Membrane voltage")

for spike_time in spikes:
    plt.axvline(spike_time, color="red", linestyle="--", alpha=0.4)

plt.xlabel("Time (ms)")
plt.ylabel("Membrane voltage (ADC units)")
plt.title("Leak-over-threshold neuron")
plt.ylim(100, 800)
plt.grid(True)
plt.legend()
plt.show()

print("Number of spikes:", len(spikes))

# Shut down simulator
pynn.end()

In [ ]:
import matplotlib.pyplot as plt

# Parameters (hardware DAC values)
v_leak = 1000
v_threshold = 300
v_reset = 400
i_bias_leak = 150

# Initialize BrainScaleS-2
pynn.setup()

# Create and configure one neuron
pop = pynn.Population(
    1,
    pynn.cells.HXNeuron(
        leak_v_leak=v_leak,
        leak_i_bias=i_bias_leak,
        threshold_v_threshold=v_threshold,
        reset_v_reset=v_reset,
        membrane_capacitance_capacitance=63,
        refractory_period_refractory_time=255,
        threshold_enable=True,
        reset_i_bias=1022,
        reset_enable_multiplication=True,
    ),
)

# Record membrane voltage and spikes
pop.record(["v", "spikes"])

# Run experiment
# pynn.run(0.3)

pop.set(constant_current_i_offset=300)

pop.set(constant_current_enable=True)
pynn.run(0.35, pynn.RunCommand.APPEND)

pop.set(constant_current_enable=True)
pynn.run(0.25, pynn.RunCommand.APPEND)

# Execute both scheduled segments
pynn.run(None, pynn.RunCommand.EXECUTE)

# Stimulation protocol
# pop.set(constant_current_enable=False)
# pynn.run(0.05, pynn.RunCommand.APPEND)

# pop.set(constant_current_enable=True)
# pynn.run(0.05, pynn.RunCommand.APPEND)

# pop.set(constant_current_enable=False)
# pynn.run(0.05, pynn.RunCommand.APPEND)

# pop.set(constant_current_enable=True)
# pynn.run(0.05, pynn.RunCommand.APPEND)

# pop.set(constant_current_enable=False)
# pynn.run(0.1, pynn.RunCommand.EXECUTE)

# Retrieve recorded data before pynn.end()
data = pop.get_data()
segment = data.segments[0]

# Membrane voltage
v = segment.filter(name="v")[0]
t = v.times.rescale("ms").magnitude
voltage = v.magnitude[:, 0]

# Spike times
spikes = segment.spiketrains[0].rescale("ms").magnitude

# Plot
plt.figure(figsize=(10, 4))
plt.plot(t, voltage, label="Membrane voltage")

for spike_time in spikes:
    plt.axvline(spike_time, color="red", linestyle="--", alpha=0.4)

plt.xlabel("Time (ms)")
plt.ylabel("Membrane voltage (ADC units)")
plt.title("Leak-over-threshold neuron")
plt.ylim(100, 800)
plt.grid(True)
plt.legend()
plt.show()

print("Number of spikes:", len(spikes))

# Shut down simulator
pynn.end()

In [ ]:
import matplotlib.pyplot as plt

# Parameters (hardware DAC values)
v_leak = 1000
v_threshold = 600
v_reset = 400
i_bias_leak = 100

# Initialize BrainScaleS-2
pynn.setup()

# Create and configure one neuron
pop = pynn.Population(
    1,
    pynn.cells.HXNeuron(
        leak_v_leak=v_leak,
        leak_i_bias=i_bias_leak,
        threshold_v_threshold=v_threshold,
        reset_v_reset=v_reset,
        membrane_capacitance_capacitance=63,
        refractory_period_refractory_time=255,
        threshold_enable=True,
        reset_i_bias=1022,
        reset_enable_multiplication=True,
    ),
)

pop.set(constant_current_i_offset=1000)

pop[0:1].record(["v", "spikes"])

pop[0:1].set(constant_current_enable=False)
pynn.run(0.35, pynn.RunCommand.APPEND)

pop[0:1].set(constant_current_enable=True)
pynn.run(0.25, pynn.RunCommand.APPEND)

pop[0:1].set(constant_current_enable=False)
pynn.run(0.35, pynn.RunCommand.APPEND)

# Execute both configurations
pynn.run(None, pynn.RunCommand.EXECUTE)

import matplotlib.pyplot as plt

data = pop.get_data()
segment = data.segments[0]

signals = segment.irregularlysampledsignals

print("Number of voltage signals:", len(signals))

plt.figure(figsize=(12, 4))

for i, signal in enumerate(signals):
    t = signal.times.rescale("ms").magnitude
    v = signal.magnitude[:, 0]

    print(f"Signal {i}: {t[0]:.3f} – {t[-1]:.3f} ms")

    plt.plot(t, v, label=f"Snippet {i}")

plt.xlabel("Time (ms)")
plt.ylabel("Membrane voltage (ADC units)")
plt.legend()
plt.grid(True)
plt.show()

pynn.end()

In [ ]:
import matplotlib.pyplot as plt
import pynn_brainscales.brainscales2 as pynn

# ==========================================
# Neuron parameters (hardware DAC values)
# ==========================================

v_leak = 700
v_threshold = 620
v_reset = 300
i_bias_leak = 100

# Exponential spike initiation
v_exp = 580
i_bias_exp = 100

# Adaptation
i_bias_a = 10  # Subthreshold adaptation
i_bias_b = 200  # Spike-triggered adaptation
i_bias_tau = 100  # Adaptation time constant control

# Current injection
i_inj = 1000

# ==========================================
# Initialize BrainScaleS-2
# ==========================================

pynn.setup()

pop = pynn.Population(
    1,
    pynn.cells.HXNeuron(
        # Membrane
        leak_v_leak=v_leak,
        leak_i_bias=i_bias_leak,
        leak_enable_division=True,
        membrane_capacitance_capacitance=63,
        # Threshold and reset
        threshold_v_threshold=v_threshold,
        threshold_enable=True,
        reset_v_reset=v_reset,
        reset_i_bias=1022,
        reset_enable_multiplication=True,
        # Refractory period
        refractory_period_refractory_time=30,
        refractory_period_enable_pause=True,
        # Exponential dynamics
        exponential_enable=True,
        exponential_v_exp=v_exp,
        exponential_i_bias=i_bias_exp,
        # Adaptation dynamics
        adaptation_enable=True,
        adaptation_enable_pulse=True,
        adaptation_i_bias_a=i_bias_a,
        adaptation_i_bias_b=i_bias_b,
        adaptation_i_bias_tau=i_bias_tau,
        adaptation_v_leak=500 + v_leak - 700,
        adaptation_v_ref=500,
        # Injected current
        constant_current_i_offset=i_inj,
        constant_current_enable=False,
    ),
)

# ==========================================
# Recording
# ==========================================

pop[0:1].record(["v", "spikes"])

# ==========================================
# Stimulation protocol
# ==========================================

pop.set(constant_current_enable=False)
pynn.run(0.35, pynn.RunCommand.APPEND)

pop.set(constant_current_enable=True)
pynn.run(0.25, pynn.RunCommand.APPEND)

pop.set(constant_current_enable=False)
pynn.run(0.35, pynn.RunCommand.APPEND)


pynn.run(None, pynn.RunCommand.EXECUTE)

# ==========================================
# Retrieve and plot
# ==========================================

data = pop.get_data()
segment = data.segments[0]

signals = segment.irregularlysampledsignals
spiketrains = segment.spiketrains

fig, ax = plt.subplots(figsize=(12, 4))

for i, signal in enumerate(signals):
    t = signal.times.rescale("ms").magnitude
    v = signal.magnitude[:, 0]

    ax.plot(t, v, label=f"Snippet {i}")

    print(f"Signal {i}: {t[0]:.3f} - {t[-1]:.3f} ms")

for train in spiketrains:
    for spike in train.rescale("ms").magnitude:
        ax.axvline(spike, color="red", linestyle="--", alpha=0.35)

# Current injection interval
ax.axvspan(0.35, 0.60, color="orange", alpha=0.15, label="Current ON")

ax.set_xlabel("Time (ms)")
ax.set_ylabel("Membrane voltage (ADC units)")
ax.set_title("BrainScaleS-2 AdEx neuron")
ax.grid(True)
ax.legend()

plt.tight_layout()
plt.show()

print("Number of spike events:", sum(len(train) for train in spiketrains))

pynn.end()

In [ ]:
import matplotlib.pyplot as plt
import pynn_brainscales.brainscales2 as pynn

# ==========================================
# Neuron parameters
# ==========================================

v_leak = 700
v_threshold = 620
v_reset = 300
i_bias_leak = 100

# Exponential dynamics
v_exp = 580
i_bias_exp = 100

# Adaptation
i_bias_a = 10
i_bias_b = 200
i_bias_tau = 100

# Different injected currents
i_inj_0 = 1000
i_inj_1 = 600
i_inj_7 = 1000

# ==========================================
# Initialize BrainScaleS-2
# ==========================================

# Two physical neurons: even (0), odd (1)
pynn.setup(neuronPermutation=[0, 1, 2, 3, 4, 5, 6, 7, 8, 9])

pop = pynn.Population(
    10,
    pynn.cells.HXNeuron(
        leak_v_leak=v_leak,
        leak_i_bias=i_bias_leak,
        leak_enable_division=True,
        membrane_capacitance_capacitance=63,
        threshold_v_threshold=v_threshold,
        threshold_enable=True,
        reset_v_reset=v_reset,
        reset_i_bias=1022,
        reset_enable_multiplication=True,
        refractory_period_refractory_time=30,
        refractory_period_enable_pause=True,
        exponential_enable=True,
        exponential_v_exp=v_exp,
        exponential_i_bias=i_bias_exp,
        adaptation_enable=True,
        adaptation_enable_pulse=True,
        adaptation_i_bias_a=i_bias_a,
        adaptation_i_bias_b=i_bias_b,
        adaptation_i_bias_tau=i_bias_tau,
        adaptation_v_leak=500 + v_leak - 700,
        adaptation_v_ref=500,
        constant_current_enable=False,
    ),
)

# Configure individual current amplitudes
pop[0].constant_current_i_offset = i_inj_0
pop[1].constant_current_i_offset = i_inj_1
pop[7].constant_current_i_offset = i_inj_7
pop[7].constant_current_i_offset = i_inj_8

# ==========================================
# Recording
# ==========================================

pop[0:2].record(["v"])
# pop[1:2].record(["v"])

pop.record(["spikes"])

# ==========================================
# Stimulation protocol
# ==========================================

# Both currents OFF
pop.set(constant_current_enable=False)
pynn.run(0.35, pynn.RunCommand.APPEND)

# Both currents ON
pop.set(constant_current_enable=True)
pynn.run(0.25, pynn.RunCommand.APPEND)

# Both currents OFF
pop.set(constant_current_enable=False)
pynn.run(0.35, pynn.RunCommand.APPEND)

# Neuron 1 current ON
pop[0:1].set(constant_current_enable=True)
pop[1:2].set(constant_current_enable=False)
pynn.run(0.25, pynn.RunCommand.APPEND)

# Both currents OFF
pop.set(constant_current_enable=False)
pynn.run(0.35, pynn.RunCommand.APPEND)

# Neuron 1 current ON
pop[0:1].set(constant_current_enable=True)
pop[1:2].set(constant_current_enable=False)
pynn.run(0.25, pynn.RunCommand.APPEND)


# Both currents OFF
pop.set(constant_current_enable=False)
pynn.run(0.35, pynn.RunCommand.APPEND)

# Neuron 1 current ON
pop[0:1].set(constant_current_enable=True)
pop[1:2].set(constant_current_enable=False)
pynn.run(0.25, pynn.RunCommand.APPEND)


# Both currents OFF
pop.set(constant_current_enable=False)
pynn.run(0.35, pynn.RunCommand.APPEND)

# Execute
pynn.run(None, pynn.RunCommand.EXECUTE)

# ==========================================
# Retrieve data
# ==========================================

data = pop.get_data()
segment = data.segments[0]

signals = segment.irregularlysampledsignals
spiketrains = segment.spiketrains

print("Voltage signals:", len(signals))
print("Spike trains:", len(spiketrains))

# ==========================================
# Plot both neurons
# ==========================================

fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)

# Map signals to neuron IDs using Neo annotations
for signal in signals:
    if signal.name != "v":
        continue

    neuron_ids = signal.array_annotations.get("channel_index", None)

    if neuron_ids is None:
        neuron_ids = signal.array_annotations.get("source_index", None)

    if neuron_ids is None:
        print("Skipping signal with unknown neuron ID")
        continue

    for col, neuron_id in enumerate(neuron_ids):
        neuron_id = int(neuron_id)

        if neuron_id not in (0, 1):
            continue

        t = signal.times.rescale("ms").magnitude
        voltage = signal.magnitude[:, col]

        axes[neuron_id].plot(t, voltage, label=f"Neuron {neuron_id}")

# Plot spikes for each neuron
for neuron_id, train in enumerate(spiketrains):
    if neuron_id >= 2:
        break

    for spike in train.rescale("ms").magnitude:
        axes[neuron_id].axvline(spike, color="red", linestyle="--", alpha=0.4)

# Plot styling
for i, ax in enumerate(axes):
    ax.axvspan(0.35, 0.60, color="orange", alpha=0.15, label="Current ON")

    ax.set_ylabel("Voltage (ADC units)")
    ax.set_title(f"Neuron {i} | I = {[i_inj_0, i_inj_1][i]}")
    ax.grid(True)
    ax.legend()

axes[-1].set_xlabel("Time (ms)")

plt.tight_layout()
plt.show()

neuron_ids = [int(cell) for cell in pop.all_cells]
raster = [[] for _ in neuron_ids]

# Iterate over all segments and all spike trains
for segment in data.segments:
    for train in segment.spiketrains:
        source_id = int(train.annotations["source_id"])
        neuron_index = neuron_ids.index(source_id)

        times = train.times.rescale("ms").magnitude
        raster[neuron_index].extend(times)

raster = [np.sort(times) for times in raster]

plt.figure(figsize=(12, 5))
plt.eventplot(raster, colors="black")
plt.xlabel("Time (ms)")
plt.ylabel("Neuron index")
plt.yticks(range(len(neuron_ids)))
plt.show()


pynn.end()